# Word2Vec — Embeddings de phrases
## Pipeline complet : Prétraitement → Word2Vec → Moyenne des vecteurs

> **Objectif** : Transformer chaque tweet en un vecteur numérique de 100 dimensions  
> représentant son sens sémantique, prêt pour un modèle de classification.


In [2]:
!pip install simplemma

   ---------------------------------------- 0.0/67.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/67.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/67.2 MB ? eta -:--:--
   ---------------------------------------- 0.8/67.2 MB 1.6 MB/s eta 0:00:41
    --------------------------------------- 1.3/67.2 MB 2.2 MB/s eta 0:00:30
   - -------------------------------------- 1.8/67.2 MB 2.4 MB/s eta 0:00:27
   - -------------------------------------- 2.4/67.2 MB 2.5 MB/s eta 0:00:26
   - -------------------------------------- 2.9/67.2 MB 2.3 MB/s eta 0:00:28
   - -------------------------------------- 3.1/67.2 MB 2.3 MB/s eta 0:00:28
   -- ------------------------------------- 3.7/67.2 MB 2.3 MB/s eta 0:00:28
   -- ------------------------------------- 3.7/67.2 MB 2.3 MB/s eta 0:00:28
   -- ------------------------------------- 4.5/67.2 MB 2.1 MB/s eta 0:00:30
   -- ------------------------------------- 4.7/67.2 MB 2.2 MB/s eta 0:00:29
   --- -------------

In [4]:
!pip install gensim

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
    --------------------------------------- 0.5/24.4 MB 198.5 kB/s eta 0:02:01
    -------

In [5]:
import pandas as pd
import numpy as np
import re
import simplemma
from gensim.models import Word2Vec
import matplotlib.pyplot as plt


## 1. Stopwords (liste NLTK intégrée)

In [6]:
STOP_WORDS = {
    'i','me','my','myself','we','our','ours','ourselves','you','your','yours',
    'yourself','yourselves','he','him','his','himself','she','her','hers',
    'herself','it','its','itself','they','them','their','theirs','themselves',
    'what','which','who','whom','this','that','these','those','am','is','are',
    'was','were','be','been','being','have','has','had','having','do','does',
    'did','doing','a','an','the','and','but','if','or','because','as','until',
    'while','of','at','by','for','with','about','against','between','into',
    'through','during','before','after','above','below','to','from','up','down',
    'in','out','on','off','over','under','again','further','then','once','here',
    'there','when','where','why','how','all','both','each','few','more','most',
    'other','some','such','no','nor','not','only','own','same','so','than',
    'too','very','s','t','can','will','just','don','should','now','d','ll',
    'm','o','re','ve','y','ain','aren','couldn','didn','doesn','hadn','hasn',
    'haven','isn','ma','mightn','mustn','needn','shan','shouldn','wasn',
    'weren','won','wouldn'
}

print(f"{len(STOP_WORDS)} stopwords chargés")


152 stopwords chargés


## 2. Fonctions de prétraitement (identiques au notebook)

In [7]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize_text(text):
    return text.split()

def remove_stopwords(tokens):
    return [w for w in tokens if w not in STOP_WORDS]

def lemmatize_text(tokens):
    return [simplemma.lemmatize(w, lang='en') for w in tokens]

def full_preprocess(text):
    if pd.isna(text):
        return []
    text = clean_text(text)
    tokens = tokenize_text(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize_text(tokens)
    return tokens

# Test rapide
print("Test : 'You are STUPID!!! 😡 123'")
print(full_preprocess("You are STUPID!!! 😡 123"))


Test : 'You are STUPID!!! 😡 123'
['stupid']


## 3. Chargement et prétraitement des données

In [8]:
df = pd.read_csv("hatespeech2.csv")

print(f"Shape : {df.shape}")
print(f"\nDistribution des classes :")
print(df['class'].value_counts().sort_index().rename({0:'Hate Speech', 1:'Offensant', 2:'Neutre'}))


FileNotFoundError: [Errno 2] No such file or directory: 'hatespeech2.csv'

In [ ]:
df['tokens'] = df['tweet'].apply(full_preprocess)

# Aperçu
pd.set_option('display.max_colwidth', 80)
df[['tweet', 'tokens']].head(5)


## 4. Entraînement du modèle Word2Vec

| Paramètre | Valeur | Description |
|-----------|--------|-------------|
| `vector_size` | 100 | Dimension des embeddings |
| `window` | 5 | Contexte de ±5 mots |
| `min_count` | 2 | Ignore les mots rares |
| `sg` | 1 | Skip-Gram |
| `epochs` | 10 | Passes d'entraînement |


In [ ]:
sentences = df['tokens'].tolist()

model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10,
    sg=1  # Skip-Gram
)

print(f"Vocabulaire : {len(model.wv)} mots")
print(f"Dimension des vecteurs : {model.vector_size}")


In [ ]:
# Mots sémantiquement proches
for word in ['hate', 'stupid', 'love', 'woman']:
    if word in model.wv:
        similars = [w for w, _ in model.wv.most_similar(word, topn=5)]
        print(f"  '{word}' → {similars}")


## 5. Embedding de phrase — Moyenne des vecteurs

Pour chaque phrase, on calcule :

$$\vec{phrase} = \frac{1}{N} \sum_{i=1}^{N} \vec{w_i}$$

Les mots absents du vocabulaire Word2Vec sont ignorés.  
Si **aucun mot** n'est connu, on retourne un vecteur nul.


In [ ]:
def sentence_embedding(tokens, model, dim=100):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if not vectors:
        return np.zeros(dim)
    return np.mean(vectors, axis=0)

embeddings = np.array([sentence_embedding(t, model) for t in df['tokens']])

print(f"Shape de la matrice d'embeddings : {embeddings.shape}")
print(f"→ {embeddings.shape[0]} phrases × {embeddings.shape[1]} dimensions")
print(f"\nExemple vecteur[0] (10 premières valeurs) :")
print(embeddings[0][:10].round(4))


## 6. Visualisation de la distribution des embeddings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribution des normes
norms = np.linalg.norm(embeddings, axis=1)
axes[0].hist(norms, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title("Distribution des normes des embeddings")
axes[0].set_xlabel("Norme L2")
axes[0].set_ylabel("Nombre de tweets")

# Moyenne par classe
class_names = {0: 'Hate Speech', 1: 'Offensant', 2: 'Neutre'}
colors = ['#e74c3c', '#e67e22', '#2ecc71']
class_means = []
for cls in sorted(df['class'].unique()):
    idx = df['class'] == cls
    class_means.append(embeddings[idx].mean(axis=0))

for i, (cls, mean_vec) in enumerate(zip(sorted(df['class'].unique()), class_means)):
    axes[1].plot(mean_vec[:30], label=class_names[cls], color=colors[i], alpha=0.8)
axes[1].set_title("Vecteur moyen par classe (30 premières dimensions)")
axes[1].set_xlabel("Dimension")
axes[1].set_ylabel("Valeur moyenne")
axes[1].legend()

plt.tight_layout()
plt.savefig("embeddings_viz.png", dpi=150, bbox_inches='tight')
plt.show()
print("Visualisation sauvegardée.")


## 7. Sauvegarde

In [ ]:
# Matrice embeddings + labels
embed_df = pd.DataFrame(embeddings, columns=[f'dim_{i}' for i in range(100)])
embed_df.insert(0, 'class', df['class'].values)
embed_df.insert(1, 'cleaned_text', df['tokens'].apply(lambda t: ' '.join(t)))

embed_df.to_csv("hatespeech_word2vec_embeddings.csv", index=False)

# Modèle Word2Vec réutilisable
model.save("word2vec_hatespeech.model")

print(f"✓ hatespeech_word2vec_embeddings.csv  ({embed_df.shape[0]} lignes × {embed_df.shape[1]} colonnes)")
print(f"✓ word2vec_hatespeech.model")
print(f"\nLes embeddings sont prêts pour la classification (SVM, Random Forest, etc.)")


## 8. Utilisation pour la classification

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = embeddings
y = df['class'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['Hate Speech', 'Offensant', 'Neutre']))
```
